<a href="https://colab.research.google.com/github/ojassahu29/Spam-Mail-Classifier/blob/main/Task2_OjasSahu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas scikit-learn nltk

import pandas as pd
import numpy as np
import nltk
import re
import urllib.request
import zipfile

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
zip_path = "spam_dataset.zip"

urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

df = pd.read_csv('SMSSpamCollection', sep='\t', names=['label', 'message'])

print("Dataset Loaded:")
print(df.head())

stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

df['clean_message'] = df['message'].apply(preprocess)

df['label'] = df['label'].map({'ham': 0, 'spam': 1})

vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(df['clean_message']).toarray()
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)

nb_preds = nb_model.predict(X_test)
lr_preds = lr_model.predict(X_test)

print("\n Naive Bayes ")
print("Accuracy:", accuracy_score(y_test, nb_preds))
print(classification_report(y_test, nb_preds))
print(confusion_matrix(y_test, nb_preds))

print("\n Logistic Regression ")
print("Accuracy:", accuracy_score(y_test, lr_preds))
print(classification_report(y_test, lr_preds))
print(confusion_matrix(y_test, lr_preds))

def predict_spam(text):
    processed_text = preprocess(text)
    vec = vectorizer.transform([processed_text]).toarray()
    lr_pred = lr_model.predict(vec)[0]
    lr_proba = lr_model.predict_proba(vec)[0]
    nb_pred = nb_model.predict(vec)[0]
    nb_proba = nb_model.predict_proba(vec)[0]
    return {
        "preprocessed_text": processed_text,
        "lr_prediction": "SPAM" if lr_pred == 1 else "HAM",
        "lr_probability": lr_proba.tolist(),
        "nb_prediction": "SPAM" if nb_pred == 1 else "HAM",
        "nb_probability": nb_proba.tolist()
    }

samples = [
    "Congratulations! You won a free iPhone. Click now!",
    "Reply to win 1000 cash!",
    "Hey, how are you doing? Let's catch up soon."
]

for i, s in enumerate(samples, 1):
    result = predict_spam(s)
    print(f"\nSample {i}: '{s}'")
    print(f"Preprocessed: '{result['preprocessed_text']}'")
    print(f"Logistic Regression Prediction: {result['lr_prediction']}")
    print(f"LR Probability (HAM, SPAM): {result['lr_probability']}")
    print(f"Naive Bayes Prediction: {result['nb_prediction']}")
    print(f"NB Probability (HAM, SPAM): {result['nb_probability']}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Dataset Loaded:
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

 Naive Bayes 
Accuracy: 0.9856502242152466
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       0.99      0.90      0.94       149

    accuracy                           0.99      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.99      0.99      0.99      1115

[[965   1]
 [ 15 134]]

 Logistic Regression 
Accuracy: 0.9704035874439462
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       966
           1       0.99      0.79      0.88       149

    accuracy        